# Oscar Wilde Multi-Voice Storyteller

This standalone Google Colab notebook turns **The Happy Prince and Other Tales** into narrated excerpts and document-grounded retellings. It discovers story boundaries and characters from the PDF, assigns built-in OpenAI voices, produces a speaker-labelled script, and combines all spoken segments into one MP3.

The notebook does not provide generic question answering or image generation. Live preview/audio tests are disabled by default to prevent accidental API usage. The generated narration uses AI-generated voices.

## 1. Install dependencies

In [ ]:
%pip install -q -U openai pypdf pydantic pydub ipywidgets numpy

## 2. Imports and configuration

In [ ]:
import hashlib
import html
import json
import os
import re
import shutil
import tempfile
import time
import uuid
from pathlib import Path
from typing import Any, Literal, Sequence

import ipywidgets as widgets
import numpy as np
from IPython.display import Audio, HTML, display
from openai import OpenAI
from pydantic import BaseModel, ConfigDict
from pydub import AudioSegment, effects
from pypdf import PdfReader

# Cost-conscious model with Responses API Structured Outputs support.
TEXT_MODEL = "gpt-4o-mini"
EMBEDDING_MODEL = "text-embedding-3-small"
SPEECH_MODEL = "tts-1"
PDF_LIMIT_BYTES = 10 * 1024 * 1024
DEFAULT_MAX_MINUTES = 5
MAX_ALLOWED_MINUTES = 15
WORDS_PER_MINUTE = 130
MAX_STORY_CONTEXT_CHARS = 50_000
MAX_TTS_SEGMENT_CHARS = 1_800
PAUSE_BETWEEN_SPEAKERS_MS = 350

GENERATED_DIR = Path("/content/generated")
GENERATED_DIR.mkdir(parents=True, exist_ok=True)

PDF_CANDIDATES = (
    Path("/sample_date/The_Happy_Prince_and_Other_Tales_by_Oscar_Wilde.pdf"),
    Path("/content/sample_data/The_Happy_Prince_and_Other_Tales_by_Oscar_Wilde.pdf"),
    Path("/content/sample_data/The_Happy_Prince,_and_Other_Tales_by_Oscar_Wilde.pdf"),
    Path("sample_data/The_Happy_Prince_and_Other_Tales_by_Oscar_Wilde.pdf"),
    Path("sample_data/The_Happy_Prince,_and_Other_Tales_by_Oscar_Wilde.pdf"),
)

# Optional correction: {"Story title": (start_page, end_page)} using one-based pages.
STORY_BOUNDARY_OVERRIDES: dict[str, tuple[int, int]] = {}
DEBUG_KEEP_SEGMENTS = False
print("Storyteller configuration loaded. Output directory:", GENERATED_DIR)

## 3. Load secrets securely

Only OPENAI_API_KEY is required. It is loaded from Google Colab Secrets first, then the environment. Optional OPENAI_CHARACTER_VOICES_JSON may map canonical character names to supported built-in OpenAI voice names. Secret values are never printed.

In [ ]:
def load_secret(name: str, *, required: bool = False) -> str | None:
    """Load a secret from Google Colab Secrets, then from the environment."""
    value: str | None = None
    try:
        from google.colab import userdata

        value = userdata.get(name)
    except ImportError:
        value = None
    except Exception:
        value = None
    value = value or os.getenv(name)
    if required and not value:
        raise RuntimeError(
            f"Missing {name}. Add it to Colab Secrets or set it as an environment variable."
        )
    return value


OPENAI_API_KEY = load_secret("OPENAI_API_KEY", required=True)
openai_client = OpenAI(api_key=OPENAI_API_KEY)
print("OpenAI credentials loaded (value hidden). The same client will generate speech.")

## 4. Validate and extract the PDF

In [ ]:
def resolve_pdf_path(candidates: Sequence[Path]) -> Path:
    """Resolve the first existing PDF candidate."""
    for candidate in candidates:
        if candidate.expanduser().is_file():
            return candidate.expanduser().resolve()
    attempted = "\n".join(f"- {path}" for path in candidates)
    raise FileNotFoundError("PDF not found. Tried:\n" + attempted)


def normalize_extracted_text(text: str) -> str:
    """Clean extraction artifacts while preserving paragraphs and dialogue punctuation."""
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = re.sub(r"(?<=\w)-\n(?=\w)", "", text)
    cleaned_lines = [re.sub(r"[ \t]+", " ", line).strip() for line in text.splitlines()]
    result: list[str] = []
    blank = False
    for line in cleaned_lines:
        if line:
            result.append(line)
            blank = False
        elif result and not blank:
            result.append("")
            blank = True
    return "\n".join(result).strip()


def validate_and_extract_pdf(pdf_path: Path) -> tuple[list[dict[str, Any]], float]:
    """Validate the PDF and return ordered page records with one-based page numbers."""
    if not pdf_path.is_file():
        raise FileNotFoundError(f"PDF does not exist: {pdf_path}")
    if pdf_path.suffix.lower() != ".pdf":
        raise ValueError("The source must be a PDF file.")
    size_bytes = pdf_path.stat().st_size
    if size_bytes <= 0 or size_bytes >= PDF_LIMIT_BYTES:
        raise ValueError("The PDF must be nonempty and smaller than 10 MB.")

    try:
        reader = PdfReader(str(pdf_path))
    except Exception as exc:
        raise ValueError(f"Unable to open PDF: {exc}") from exc
    if not reader.pages:
        raise ValueError("The PDF contains no pages.")

    pages: list[dict[str, Any]] = []
    for page_number, page in enumerate(reader.pages, start=1):
        try:
            text = normalize_extracted_text(page.extract_text() or "")
        except Exception:
            text = ""
        pages.append({"page_number": page_number, "text": text})
    if not any(page["text"] for page in pages):
        raise ValueError("The PDF has no extractable text and may require OCR.")

    size_mb = size_bytes / (1024 ** 2)
    print(f"Resolved PDF: {pdf_path}")
    print(f"Validated: {size_mb:.2f} MB, {len(pages)} pages")
    return pages, size_mb


PDF_PATH = resolve_pdf_path(PDF_CANDIDATES)
PDF_PAGES, PDF_SIZE_MB = validate_and_extract_pdf(PDF_PATH)
PDF_FINGERPRINT = hashlib.sha256(PDF_PATH.read_bytes()).hexdigest()[:16]
print("Document fingerprint:", PDF_FINGERPRINT)

## 5. Structured data models

In [ ]:
VoiceRole = Literal["narrator", "adult_male", "adult_female", "child", "creature", "neutral"]
StoryMode = Literal["excerpt", "retelling"]


class StrictModel(BaseModel):
    model_config = ConfigDict(extra="forbid")


class StoryBoundary(StrictModel):
    title: str
    start_page: int
    end_page: int


class StoryBoundaryList(StrictModel):
    stories: list[StoryBoundary]


class CharacterProfile(StrictModel):
    name: str
    aliases: list[str]
    voice_role: VoiceRole
    evidence: str


class StoryCatalog(StrictModel):
    story_title: str
    start_page: int
    end_page: int
    characters: list[CharacterProfile]


class CharacterExtraction(StrictModel):
    characters: list[CharacterProfile]


class StorySelection(StrictModel):
    story_title: str
    focus: str
    complete_story: bool


class NarrationSegment(StrictModel):
    speaker: str
    voice_role: VoiceRole
    text: str
    source_pages: list[int]


class NarrationScript(StrictModel):
    story_title: str
    mode: StoryMode
    label: str
    segments: list[NarrationSegment]
    source_pages: list[int]


VALID_VOICE_ROLES = {"narrator", "adult_male", "adult_female", "child", "creature", "neutral"}
print("Structured models ready.")

## 6. Discover stories and extract characters

The page map contains only short page previews, not the entire PDF. Story boundaries are validated and may be corrected with `STORY_BOUNDARY_OVERRIDES`. Character roles are grounded in each complete story and cached using the PDF fingerprint.

In [ ]:
def build_page_map(pages: Sequence[dict[str, Any]], preview_chars: int = 1600) -> str:
    """Build a compact page-numbered map for story-boundary discovery."""
    return "\n\n".join(
        f"[PAGE {page['page_number']}]\n{page['text'][:preview_chars]}"
        for page in pages
        if page["text"]
    )


def validate_story_boundaries(
    boundaries: Sequence[StoryBoundary], total_pages: int
) -> list[StoryBoundary]:
    """Normalize boundaries and enforce ordered, non-overlapping page ranges."""
    if not boundaries:
        raise ValueError("No story boundaries were discovered.")
    normalized: list[StoryBoundary] = []
    seen_titles: set[str] = set()
    for original in sorted(boundaries, key=lambda item: (item.start_page, item.title.casefold())):
        title = re.sub(r"\s+", " ", original.title).strip()
        if not title or title.casefold() in seen_titles:
            continue
        start, end = original.start_page, original.end_page
        if title in STORY_BOUNDARY_OVERRIDES:
            start, end = STORY_BOUNDARY_OVERRIDES[title]
        if not (1 <= start <= end <= total_pages):
            raise ValueError(f"Invalid page range for {title}: {start}-{end}")
        if normalized and start <= normalized[-1].end_page:
            raise ValueError(
                f"Overlapping story boundaries: {normalized[-1].title} and {title}. "
                "Use STORY_BOUNDARY_OVERRIDES to correct them."
            )
        normalized.append(StoryBoundary(title=title, start_page=start, end_page=end))
        seen_titles.add(title.casefold())
    if not normalized:
        raise ValueError("No valid story boundaries remain after validation.")
    return normalized


def discover_story_boundaries() -> list[StoryBoundary]:
    """Use Structured Outputs to identify the tales and their page ranges."""
    try:
        response = openai_client.responses.parse(
            model=TEXT_MODEL,
            instructions=(
                "Identify only the complete fairy tales in this Oscar Wilde collection, not the title "
                "page, contents page, publication notes, headers, or footers. Return each tale's exact "
                "title and inclusive one-based start/end PDF pages. Order stories by their real start. "
                "Ranges must not overlap. A page preview may contain the end of one story and the title "
                "of the next; assign the shared page to the story with most of its text and avoid overlap."
            ),
            input=build_page_map(PDF_PAGES),
            text_format=StoryBoundaryList,
        )
    except Exception as exc:
        raise RuntimeError(f"Story-boundary discovery failed: {exc}") from exc
    if response.output_parsed is None:
        raise RuntimeError("Story-boundary discovery returned no structured result.")
    return validate_story_boundaries(response.output_parsed.stories, len(PDF_PAGES))


def story_page_records(boundary: StoryBoundary) -> list[dict[str, Any]]:
    """Return page records belonging to one validated story boundary."""
    return [
        page
        for page in PDF_PAGES
        if boundary.start_page <= int(page["page_number"]) <= boundary.end_page and page["text"]
    ]


def story_text(boundary: StoryBoundary) -> str:
    """Return page-labelled text for one story."""
    records = story_page_records(boundary)
    if not records:
        raise ValueError(f"No text found for story: {boundary.title}")
    return "\n\n".join(
        f"[PAGE {page['page_number']}]\n{page['text']}" for page in records
    )


def normalize_character_catalog(
    boundary: StoryBoundary, extracted: CharacterExtraction
) -> StoryCatalog:
    """Merge duplicate aliases and guarantee a narrator profile."""
    profiles: list[CharacterProfile] = []
    claimed: set[str] = set()
    for profile in extracted.characters:
        name = re.sub(r"\s+", " ", profile.name).strip()
        if not name or name.casefold() in claimed or name.casefold() == "narrator":
            continue
        aliases: list[str] = []
        for alias in [name, *profile.aliases]:
            alias = re.sub(r"\s+", " ", alias).strip()
            key = alias.casefold()
            if alias and key not in claimed and key not in {item.casefold() for item in aliases}:
                aliases.append(alias)
        claimed.update(alias.casefold() for alias in aliases)
        profiles.append(
            CharacterProfile(
                name=name,
                aliases=[alias for alias in aliases if alias.casefold() != name.casefold()],
                voice_role=profile.voice_role if profile.voice_role in VALID_VOICE_ROLES else "neutral",
                evidence=re.sub(r"\s+", " ", profile.evidence).strip()[:300],
            )
        )
    profiles.insert(
        0,
        CharacterProfile(
            name="Narrator",
            aliases=[],
            voice_role="narrator",
            evidence="Narrates prose outside attributed character dialogue.",
        ),
    )
    return StoryCatalog(
        story_title=boundary.title,
        start_page=boundary.start_page,
        end_page=boundary.end_page,
        characters=profiles,
    )


def extract_story_characters(boundary: StoryBoundary) -> StoryCatalog:
    """Extract canonical characters, aliases, evidence, and suggested voice roles."""
    try:
        response = openai_client.responses.parse(
            model=TEXT_MODEL,
            instructions=(
                "Extract characters from exactly one supplied Oscar Wilde story. Use canonical names "
                "and merge titles, descriptions, and aliases that refer to the same character. Exclude "
                "the author, publishers, story titles, and generic groups that never speak or act. Infer "
                "a conservative voice role from explicit textual evidence: narrator, adult_male, "
                "adult_female, child, creature, or neutral. Use neutral when unclear. Evidence must be a "
                "brief paraphrase, not a long quotation. Do not add a narrator; the application does that."
            ),
            input=f"Story title: {boundary.title}\n\n{story_text(boundary)}",
            text_format=CharacterExtraction,
        )
    except Exception as exc:
        raise RuntimeError(f"Character extraction failed for {boundary.title}: {exc}") from exc
    if response.output_parsed is None:
        raise RuntimeError(f"No character catalog returned for {boundary.title}.")
    return normalize_character_catalog(boundary, response.output_parsed)


CATALOG_CACHE_PATH = GENERATED_DIR / f"story_catalog_{PDF_FINGERPRINT}.json"


def build_story_catalog(force: bool = False) -> dict[str, StoryCatalog]:
    """Build or load the fingerprinted story and character catalog."""
    if CATALOG_CACHE_PATH.is_file() and not force:
        try:
            payload = json.loads(CATALOG_CACHE_PATH.read_text(encoding="utf-8"))
            if payload.get("pdf_fingerprint") == PDF_FINGERPRINT:
                catalogs = [StoryCatalog.model_validate(item) for item in payload["catalogs"]]
                print(f"Loaded {len(catalogs)} story catalogs from cache.")
                return {catalog.story_title: catalog for catalog in catalogs}
        except Exception as exc:
            print("Ignoring invalid story-catalog cache:", exc)

    boundaries = discover_story_boundaries()
    catalogs: list[StoryCatalog] = []
    for index, boundary in enumerate(boundaries, start=1):
        print(f"Extracting characters {index}/{len(boundaries)}: {boundary.title}", flush=True)
        catalogs.append(extract_story_characters(boundary))
    payload = {
        "pdf_fingerprint": PDF_FINGERPRINT,
        "catalogs": [catalog.model_dump(mode="json") for catalog in catalogs],
    }
    CATALOG_CACHE_PATH.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"Cached {len(catalogs)} story catalogs (custom voice overrides are not stored).")
    return {catalog.story_title: catalog for catalog in catalogs}


STORY_CATALOGS = build_story_catalog()
for title, catalog in STORY_CATALOGS.items():
    print(f"- {title}: pages {catalog.start_page}-{catalog.end_page}, "
          f"{len(catalog.characters) - 1} detected character(s)")

## 7. Build and cache the semantic story index

The index is used only to locate a requested scene inside a selected story. It stores ordered story units and explicit OpenAI embeddings in fingerprinted cache files; it is not a generic RAG assistant.

In [ ]:
SENTENCE_SPLIT = re.compile(r"(?<=[.!?])\s+(?=[A-Z\"\'\u201c\u2018])")


def split_paragraph(paragraph: str, max_chars: int = 900) -> list[str]:
    """Split long paragraphs at sentence or word boundaries."""
    paragraph = re.sub(r"\s+", " ", paragraph).strip()
    if not paragraph:
        return []
    if len(paragraph) <= max_chars:
        return [paragraph]
    sentences = [part.strip() for part in SENTENCE_SPLIT.split(paragraph) if part.strip()]
    units: list[str] = []
    current: list[str] = []
    for sentence in sentences:
        if len(sentence) > max_chars:
            words = sentence.split()
            for word in words:
                candidate = " ".join([*current, word])
                if current and len(candidate) > max_chars:
                    units.append(" ".join(current))
                    current = [word]
                else:
                    current.append(word)
            continue
        candidate = " ".join([*current, sentence])
        if current and len(candidate) > max_chars:
            units.append(" ".join(current))
            current = [sentence]
        else:
            current.append(sentence)
    if current:
        units.append(" ".join(current))
    return units


def create_story_units() -> list[dict[str, Any]]:
    """Create ordered, page-preserving units that never cross story boundaries."""
    units: list[dict[str, Any]] = []
    for catalog in STORY_CATALOGS.values():
        boundary = StoryBoundary(
            title=catalog.story_title,
            start_page=catalog.start_page,
            end_page=catalog.end_page,
        )
        story_unit_index = 0
        for page in story_page_records(boundary):
            paragraphs = [part for part in re.split(r"\n\s*\n", page["text"]) if part.strip()]
            for paragraph in paragraphs:
                for text in split_paragraph(paragraph):
                    units.append(
                        {
                            "global_index": len(units),
                            "story_unit_index": story_unit_index,
                            "story_title": catalog.story_title,
                            "page_number": int(page["page_number"]),
                            "text": text,
                        }
                    )
                    story_unit_index += 1
    if not units:
        raise ValueError("No story units could be created.")
    return units


def embed_texts(texts: Sequence[str], batch_size: int = 96) -> list[list[float]]:
    """Generate ordered OpenAI embeddings without displaying vectors."""
    if not texts or any(not isinstance(text, str) or not text.strip() for text in texts):
        raise ValueError("Embedding input must contain nonempty strings.")
    embeddings: list[list[float]] = []
    try:
        for start in range(0, len(texts), batch_size):
            response = openai_client.embeddings.create(
                model=EMBEDDING_MODEL,
                input=[text.replace("\n", " ") for text in texts[start : start + batch_size]],
            )
            embeddings.extend(item.embedding for item in sorted(response.data, key=lambda item: item.index))
    except Exception as exc:
        raise RuntimeError(f"Embedding generation failed: {exc}") from exc
    if len(embeddings) != len(texts):
        raise RuntimeError("Unexpected embedding count.")
    return embeddings


UNITS_CACHE_PATH = GENERATED_DIR / f"story_units_{PDF_FINGERPRINT}.json"
EMBEDDINGS_CACHE_PATH = GENERATED_DIR / f"story_embeddings_{PDF_FINGERPRINT}.npz"


def build_story_index(force: bool = False) -> tuple[list[dict[str, Any]], np.ndarray]:
    """Build or load story units and their fingerprinted embedding matrix."""
    if UNITS_CACHE_PATH.is_file() and EMBEDDINGS_CACHE_PATH.is_file() and not force:
        try:
            units = json.loads(UNITS_CACHE_PATH.read_text(encoding="utf-8"))
            with np.load(EMBEDDINGS_CACHE_PATH) as data:
                matrix = data["embeddings"]
            if len(units) == len(matrix) and len(units) > 0:
                print(f"Loaded {len(units)} semantic story units from cache.")
                return units, matrix
        except Exception as exc:
            print("Ignoring invalid embedding cache:", exc)

    units = create_story_units()
    matrix = np.asarray(embed_texts([unit["text"] for unit in units]), dtype=np.float32)
    UNITS_CACHE_PATH.write_text(json.dumps(units, ensure_ascii=False), encoding="utf-8")
    np.savez_compressed(EMBEDDINGS_CACHE_PATH, embeddings=matrix)
    print(f"Embedded and cached {len(units)} story units; vectors are not displayed.")
    return units, matrix


STORY_UNITS, STORY_EMBEDDINGS = build_story_index()

## 8. Select a story or passage

In [ ]:
def validate_request(request: str, mode: StoryMode, max_minutes: int) -> tuple[str, StoryMode, int]:
    """Validate public storyteller inputs."""
    if not isinstance(request, str) or not request.strip():
        raise ValueError("request must be a nonempty string.")
    if mode not in {"excerpt", "retelling"}:
        raise ValueError("mode must be 'excerpt' or 'retelling'.")
    if not isinstance(max_minutes, int) or not 1 <= max_minutes <= MAX_ALLOWED_MINUTES:
        raise ValueError(f"max_minutes must be between 1 and {MAX_ALLOWED_MINUTES}.")
    return request.strip(), mode, max_minutes


def parse_story_selection(request: str) -> StorySelection:
    """Resolve a natural-language request against the discovered story catalog."""
    titles = list(STORY_CATALOGS)
    response = openai_client.responses.parse(
        model=TEXT_MODEL,
        instructions=(
            "Select exactly one title from the supplied available titles. Extract the requested scene, "
            "event, character, or passage as a concise focus. Set complete_story true only when the user "
            "clearly requests the whole story; otherwise false. Do not invent a title."
        ),
        input=f"Available titles: {json.dumps(titles, ensure_ascii=False)}\nUser request: {request}",
        text_format=StorySelection,
    )
    selection = response.output_parsed
    if selection is None or selection.story_title not in STORY_CATALOGS:
        raise ValueError("The request did not resolve to one discovered story title.")
    selection.focus = selection.focus.strip() or request
    return selection


def cosine_similarities(query: Sequence[float], matrix: np.ndarray) -> np.ndarray:
    """Return cosine similarities between one vector and a matrix."""
    query_vector = np.asarray(query, dtype=np.float32)
    query_norm = np.linalg.norm(query_vector)
    matrix_norms = np.linalg.norm(matrix, axis=1)
    denominator = matrix_norms * query_norm
    return np.divide(
        matrix @ query_vector,
        denominator,
        out=np.zeros(len(matrix), dtype=np.float32),
        where=denominator != 0,
    )


def complete_story_context(story_title: str) -> tuple[str, list[int]]:
    """Return bounded, page-labelled text for a complete story."""
    catalog = STORY_CATALOGS[story_title]
    boundary = StoryBoundary(
        title=catalog.story_title,
        start_page=catalog.start_page,
        end_page=catalog.end_page,
    )
    context = story_text(boundary)
    if len(context) > MAX_STORY_CONTEXT_CHARS:
        raise ValueError(
            f"{story_title} exceeds the safe context limit. Request a specific part instead."
        )
    pages = [page["page_number"] for page in story_page_records(boundary)]
    return context, pages


def semantic_passage_context(
    story_title: str, focus: str, target_chars: int
) -> tuple[str, list[int]]:
    """Select the best unit and expand contiguously without leaving its story."""
    story_positions = [
        index for index, unit in enumerate(STORY_UNITS) if unit["story_title"] == story_title
    ]
    if not story_positions:
        raise ValueError(f"No indexed units for {story_title}.")
    query_embedding = embed_texts([focus])[0]
    local_matrix = STORY_EMBEDDINGS[story_positions]
    best_local = int(np.argmax(cosine_similarities(query_embedding, local_matrix)))
    selected_local = {best_local}
    left, right = best_local - 1, best_local + 1
    current_chars = len(STORY_UNITS[story_positions[best_local]]["text"])
    while current_chars < target_chars and (left >= 0 or right < len(story_positions)):
        candidates = []
        if left >= 0:
            candidates.append(left)
        if right < len(story_positions):
            candidates.append(right)
        chosen = max(candidates, key=lambda item: -abs(item - best_local))
        selected_local.add(chosen)
        current_chars += len(STORY_UNITS[story_positions[chosen]]["text"])
        if chosen == left:
            left -= 1
        else:
            right += 1
    selected_units = [STORY_UNITS[story_positions[index]] for index in sorted(selected_local)]
    pages = sorted({int(unit["page_number"]) for unit in selected_units})
    context = "\n\n".join(
        f"[PAGE {unit['page_number']}]\n{unit['text']}" for unit in selected_units
    )
    return context, pages


def select_story_context(
    selection: StorySelection, mode: StoryMode, max_minutes: int
) -> tuple[str, list[int]]:
    """Choose complete-story or focused contiguous context for script generation."""
    if selection.complete_story and mode == "excerpt":
        raise ValueError(
            "Exact complete-story readings are disabled by default. Request a specific passage, "
            "or use retelling mode for a condensed complete story."
        )
    if selection.complete_story:
        return complete_story_context(selection.story_title)
    target_chars = min(12_000, max(2_500, max_minutes * WORDS_PER_MINUTE * 7))
    return semantic_passage_context(selection.story_title, selection.focus, target_chars)

## 9. Prepare a speaker-labelled narration script

`excerpt` segments must be verbatim substrings of the selected normalized source. `retelling` produces a clearly labelled adaptation constrained to the selected story and its extracted characters.

In [ ]:
_SCRIPT_CACHE: dict[tuple[str, str, int], NarrationScript] = {}


def character_alias_map(catalog: StoryCatalog) -> dict[str, CharacterProfile]:
    """Map canonical names and aliases to their profiles."""
    mapping: dict[str, CharacterProfile] = {}
    for profile in catalog.characters:
        for name in [profile.name, *profile.aliases]:
            mapping[name.casefold().strip()] = profile
    return mapping


def normalize_for_source_check(text: str) -> str:
    """Normalize whitespace and quotation glyphs for excerpt containment checks."""
    translation = str.maketrans({"\u201c": '"', "\u201d": '"', "\u2018": "'", "\u2019": "'"})
    return re.sub(r"\s+", " ", text.translate(translation)).strip().casefold()


def validate_and_normalize_script(
    script: NarrationScript,
    catalog: StoryCatalog,
    mode: StoryMode,
    context: str,
    allowed_pages: Sequence[int],
) -> NarrationScript:
    """Ground speakers, roles, source pages, labels, and exact excerpt wording."""
    if not script.segments:
        raise ValueError("The narration script contains no segments.")
    aliases = character_alias_map(catalog)
    source_normalized = normalize_for_source_check(context)
    pages_set = {int(page) for page in allowed_pages}
    normalized_segments: list[NarrationSegment] = []
    for segment in script.segments:
        text = re.sub(r"\s+", " ", segment.text).strip()
        if not text:
            continue
        profile = aliases.get(segment.speaker.casefold().strip())
        if profile is None:
            profile = aliases["narrator"]
        segment_pages = sorted({int(page) for page in segment.source_pages if int(page) in pages_set})
        if not segment_pages:
            segment_pages = sorted(pages_set)
        if mode == "excerpt" and normalize_for_source_check(text) not in source_normalized:
            raise ValueError(
                f"Excerpt validation failed because a {profile.name} segment was not verbatim source text."
            )
        normalized_segments.append(
            NarrationSegment(
                speaker=profile.name,
                voice_role=profile.voice_role,
                text=text,
                source_pages=segment_pages,
            )
        )
    if not normalized_segments:
        raise ValueError("No valid narration segments remain.")
    return NarrationScript(
        story_title=catalog.story_title,
        mode=mode,
        label="Faithful PDF excerpt" if mode == "excerpt" else "Document-grounded adaptation",
        segments=normalized_segments,
        source_pages=sorted(pages_set),
    )


def preview_story(
    request: str,
    *,
    mode: Literal["excerpt", "retelling"] = "retelling",
    max_minutes: int = 5,
) -> NarrationScript:
    """Prepare and display a grounded narration script without generating speech."""
    request, mode, max_minutes = validate_request(request, mode, max_minutes)
    cache_key = (request, mode, max_minutes)
    if cache_key in _SCRIPT_CACHE:
        script = _SCRIPT_CACHE[cache_key]
        display_script(script)
        return script

    selection = parse_story_selection(request)
    context, pages = select_story_context(selection, mode, max_minutes)
    catalog = STORY_CATALOGS[selection.story_title]
    character_summary = [
        {
            "name": profile.name,
            "aliases": profile.aliases,
            "voice_role": profile.voice_role,
        }
        for profile in catalog.characters
    ]
    target_words = max_minutes * WORDS_PER_MINUTE
    if mode == "excerpt":
        instructions = (
            "Create a dramatic reading script from only the supplied PDF passage. Every segment's text "
            "must be copied verbatim as one contiguous substring of the passage, except whitespace may "
            "be normalized. Do not paraphrase, modernize, summarize, or invent words. Attribute direct "
            "speech only to listed characters when supported; assign all other prose to Narrator. Use "
            "only supplied page numbers. Keep the total near the requested word limit by selecting a "
            "coherent excerpt rather than rewriting it."
        )
    else:
        instructions = (
            "Create a concise fairy-tale narration script grounded only in the supplied PDF context. "
            "This is an adaptation, so preserve the original characters, sequence, themes, and events "
            "without adding new plot facts. Use only listed canonical speakers, with Narrator for prose. "
            "Use only supplied page numbers. Keep the total at or below the requested word limit."
        )
    response = openai_client.responses.parse(
        model=TEXT_MODEL,
        instructions=instructions,
        input=(
            f"Story: {catalog.story_title}\nMode: {mode}\nRequested focus: {selection.focus}\n"
            f"Maximum words: {target_words}\nCharacters: "
            f"{json.dumps(character_summary, ensure_ascii=False)}\n\nPDF context:\n{context}"
        ),
        text_format=NarrationScript,
    )
    if response.output_parsed is None:
        raise RuntimeError("Narration-script generation returned no structured result.")
    script = validate_and_normalize_script(
        response.output_parsed, catalog, mode, context, pages
    )
    _SCRIPT_CACHE[cache_key] = script
    display_script(script)
    return script


def display_script(script: NarrationScript) -> None:
    """Display a safe speaker-labelled transcript without exposing voice configuration."""
    blocks = [
        f"<h3>{html.escape(script.story_title)}</h3>",
        f"<p><strong>{html.escape(script.label)}</strong> | Pages "
        f"{html.escape(', '.join(map(str, script.source_pages)))}</p>",
    ]
    for segment in script.segments:
        blocks.append(
            f"<p><strong>{html.escape(segment.speaker)}:</strong> "
            f"{html.escape(segment.text)}</p>"
        )
    display(HTML("\n".join(blocks)))

## 10. Resolve voices and generate multi-voice audio

In [ ]:
OPENAI_AVAILABLE_VOICES = {
    "alloy", "ash", "ballad", "coral", "echo", "fable", "onyx",
    "nova", "sage", "shimmer", "verse", "marin", "cedar",
}
OPENAI_VOICES_BY_ROLE: dict[str, str] = {
    "narrator": "cedar",
    "adult_male": "onyx",
    "adult_female": "coral",
    "child": "shimmer",
    "creature": "fable",
    "neutral": "alloy",
}


def load_character_voice_overrides() -> dict[str, str]:
    """Load optional character-to-OpenAI-voice mappings without printing them."""
    raw_mapping = load_secret("OPENAI_CHARACTER_VOICES_JSON")
    if not raw_mapping:
        return {}
    try:
        parsed = json.loads(raw_mapping)
    except json.JSONDecodeError as exc:
        raise ValueError("OPENAI_CHARACTER_VOICES_JSON must be valid JSON.") from exc
    if not isinstance(parsed, dict):
        raise ValueError("Character voice JSON must be an object mapping names to voices.")

    overrides: dict[str, str] = {}
    for character, voice in parsed.items():
        if not isinstance(character, str) or not isinstance(voice, str):
            raise ValueError("Character names and OpenAI voice names must be strings.")
        normalized_voice = voice.strip().lower()
        if normalized_voice not in OPENAI_AVAILABLE_VOICES:
            raise ValueError(
                f"Unsupported OpenAI voice for {character}. Choose one of: "
                + ", ".join(sorted(OPENAI_AVAILABLE_VOICES))
            )
        overrides[character.casefold().strip()] = normalized_voice
    return overrides


def resolve_voice_name(
    speaker: str,
    role: str,
    character_voices: dict[str, str],
) -> str:
    """Resolve a character override, then a role voice, then the neutral voice."""
    voice = (
        character_voices.get(speaker.casefold().strip())
        or OPENAI_VOICES_BY_ROLE.get(role)
        or OPENAI_VOICES_BY_ROLE["neutral"]
    )
    if voice not in OPENAI_AVAILABLE_VOICES:
        raise ValueError(f"Unsupported OpenAI voice: {voice}")
    return voice


def split_narration_segment(
    segment: NarrationSegment, max_chars: int = MAX_TTS_SEGMENT_CHARS
) -> list[NarrationSegment]:
    """Split long TTS input while retaining speaker, role, and page metadata."""
    if len(segment.text) <= max_chars:
        return [segment]
    pieces = split_paragraph(segment.text, max_chars=max_chars)
    return [
        NarrationSegment(
            speaker=segment.speaker,
            voice_role=segment.voice_role,
            text=piece,
            source_pages=segment.source_pages,
        )
        for piece in pieces
    ]


def synthesize_segment(text: str, voice: str, output_path: Path) -> None:
    """Generate one OpenAI text-to-speech MP3 segment."""
    if voice not in OPENAI_AVAILABLE_VOICES:
        raise ValueError(f"Unsupported OpenAI voice: {voice}")
    try:
        with openai_client.audio.speech.with_streaming_response.create(
            model=SPEECH_MODEL,
            voice=voice,
            input=text,
            response_format="mp3",
        ) as response:
            response.stream_to_file(output_path)
    except Exception as exc:
        raise RuntimeError(f"OpenAI speech generation failed: {exc}") from exc
    if not output_path.is_file() or output_path.stat().st_size == 0:
        raise RuntimeError("OpenAI returned an empty audio segment.")


def safe_filename(value: str) -> str:
    """Create a short filesystem-safe filename component."""
    cleaned = re.sub(r"[^A-Za-z0-9]+", "_", value).strip("_").lower()
    return cleaned[:50] or "story"


def render_story_audio(script: NarrationScript, *, debug: bool = False) -> str:
    """Synthesize, normalize, concatenate, save, and display a multi-voice MP3."""
    if shutil.which("ffmpeg") is None:
        raise RuntimeError("FFmpeg is required by pydub but was not found in this runtime.")
    character_voices = load_character_voice_overrides()

    expanded_segments = [
        piece for segment in script.segments for piece in split_narration_segment(segment)
    ]
    if not expanded_segments:
        raise ValueError("There are no narration segments to synthesize.")

    if debug:
        work_dir = GENERATED_DIR / f"segments_{uuid.uuid4().hex[:10]}"
        work_dir.mkdir(parents=True, exist_ok=False)
        temporary_context = None
    else:
        temporary_context = tempfile.TemporaryDirectory(dir=GENERATED_DIR)
        work_dir = Path(temporary_context.name)

    combined = AudioSegment.empty()
    try:
        for index, segment in enumerate(expanded_segments, start=1):
            started = time.perf_counter()
            print(
                f"Generating segment {index}/{len(expanded_segments)}: {segment.speaker}",
                flush=True,
            )
            voice = resolve_voice_name(
                segment.speaker,
                segment.voice_role,
                character_voices,
            )
            segment_path = work_dir / f"segment_{index:04d}.mp3"
            synthesize_segment(segment.text, voice, segment_path)
            audio = effects.normalize(AudioSegment.from_file(segment_path, format="mp3"))
            combined += audio
            if index < len(expanded_segments):
                combined += AudioSegment.silent(duration=PAUSE_BETWEEN_SPEAKERS_MS)
            print(f"  completed in {time.perf_counter() - started:.1f}s", flush=True)

        output_path = GENERATED_DIR / (
            f"{safe_filename(script.story_title)}_{script.mode}_{uuid.uuid4().hex[:10]}.mp3"
        )
        combined.export(output_path, format="mp3", bitrate="128k")
    finally:
        if temporary_context is not None:
            temporary_context.cleanup()
    if debug:
        print("Debug segment directory retained:", work_dir)
    display(Audio(filename=str(output_path)))
    return str(output_path)


def tell_story(
    request: str,
    *,
    mode: Literal["excerpt", "retelling"] = "retelling",
    max_minutes: int = 5,
) -> str:
    """Prepare, display, synthesize, and return a multi-voice story MP3 path."""
    request, mode, max_minutes = validate_request(request, mode, max_minutes)
    script = preview_story(request, mode=mode, max_minutes=max_minutes)
    return render_story_audio(script, debug=DEBUG_KEEP_SEGMENTS)

## 11. Interactive Colab storyteller

In [ ]:
story_dropdown = widgets.Dropdown(
    options=list(STORY_CATALOGS),
    description="Story:",
    layout=widgets.Layout(width="95%"),
)
mode_dropdown = widgets.Dropdown(
    options=[("Grounded retelling", "retelling"), ("Faithful excerpt", "excerpt")],
    value="retelling",
    description="Mode:",
)
focus_text = widgets.Textarea(
    placeholder="Example: Tell the complete story, or narrate the scene where the Swallow meets the Prince.",
    description="Request:",
    layout=widgets.Layout(width="95%", height="100px"),
)
minutes_slider = widgets.IntSlider(
    value=DEFAULT_MAX_MINUTES,
    min=1,
    max=10,
    step=1,
    description="Minutes:",
    continuous_update=False,
)
preview_button = widgets.Button(description="Preview script", button_style="info")
audio_button = widgets.Button(description="Generate audio", button_style="success")
story_output = widgets.Output()


def widget_request() -> str:
    """Compose a natural request from widget values."""
    focus = focus_text.value.strip()
    if not focus:
        focus = "Tell the complete story" if mode_dropdown.value == "retelling" else "Read its opening passage"
    return f"From {story_dropdown.value}: {focus}"


def on_preview_clicked(_: widgets.Button) -> None:
    with story_output:
        story_output.clear_output(wait=True)
        started = time.perf_counter()
        try:
            preview_story(
                widget_request(),
                mode=mode_dropdown.value,
                max_minutes=minutes_slider.value,
            )
            print(f"Preview ready in {time.perf_counter() - started:.1f}s")
        except Exception as exc:
            print("Preview failed:", exc)


def on_audio_clicked(_: widgets.Button) -> None:
    with story_output:
        story_output.clear_output(wait=True)
        started = time.perf_counter()
        try:
            result = tell_story(
                widget_request(),
                mode=mode_dropdown.value,
                max_minutes=minutes_slider.value,
            )
            print(f"Saved: {result}")
            print(f"Total time: {time.perf_counter() - started:.1f}s")
        except Exception as exc:
            print("Audio generation failed:", exc)


preview_button.on_click(on_preview_clicked)
audio_button.on_click(on_audio_clicked)
display(
    widgets.VBox(
        [
            widgets.HTML("<h3>Oscar Wilde Multi-Voice Storyteller</h3>"),
            story_dropdown,
            mode_dropdown,
            focus_text,
            minutes_slider,
            widgets.HBox([preview_button, audio_button]),
            story_output,
        ]
    )
)

## 12. Validation and tests

Local checks below do not call TTS. Live preview and audio smoke tests are included but disabled by default. Turn them on deliberately after confirming secrets, quota, and the selected passage.

In [ ]:
def run_local_validation() -> None:
    """Run deterministic checks without additional OpenAI generation calls."""
    assert PDF_PATH.is_file() and PDF_SIZE_MB < 10
    assert STORY_CATALOGS
    ordered = sorted(STORY_CATALOGS.values(), key=lambda item: item.start_page)
    for index, catalog in enumerate(ordered):
        assert 1 <= catalog.start_page <= catalog.end_page <= len(PDF_PAGES)
        if index:
            assert ordered[index - 1].end_page < catalog.start_page
        assert catalog.characters[0].name == "Narrator"
        aliases = []
        for profile in catalog.characters:
            assert profile.voice_role in VALID_VOICE_ROLES
            aliases.extend([profile.name.casefold(), *(alias.casefold() for alias in profile.aliases)])
        assert len(aliases) == len(set(aliases)), f"Duplicate aliases in {catalog.story_title}"

    assert len(STORY_UNITS) == len(STORY_EMBEDDINGS) > 0
    assert all(unit["story_title"] in STORY_CATALOGS for unit in STORY_UNITS)
    assert resolve_voice_name("A", "child", {}) == "shimmer"
    assert resolve_voice_name("A", "child", {"a": "nova"}) == "nova"
    assert resolve_voice_name("A", "neutral", {}) == "alloy"

    long_segment = NarrationSegment(
        speaker="Narrator",
        voice_role="narrator",
        text=("A short sentence. " * 200).strip(),
        source_pages=[1],
    )
    pieces = split_narration_segment(long_segment, max_chars=300)
    assert len(pieces) > 1 and all(len(piece.text) <= 300 for piece in pieces)
    assert all(piece.speaker == "Narrator" for piece in pieces)

    source = '[PAGE 1]\nHe said, "Good morning."'
    assert normalize_for_source_check('He said, "Good morning."') in normalize_for_source_check(source)
    assert PDF_FINGERPRINT in CATALOG_CACHE_PATH.name
    assert PDF_FINGERPRINT in EMBEDDINGS_CACHE_PATH.name
    print("[PASS] Local storyteller validation passed.")


run_local_validation()

In [ ]:
RUN_LIVE_PREVIEW_TEST = False
RUN_LIVE_AUDIO_SMOKE_TEST = False

sample_title = next(iter(STORY_CATALOGS))
sample_request = f"From {sample_title}, narrate a short opening scene."

if RUN_LIVE_PREVIEW_TEST:
    started = time.perf_counter()
    try:
        preview = preview_story(sample_request, mode="excerpt", max_minutes=1)
        assert isinstance(preview, NarrationScript) and preview.segments
        print(f"[PASS] Live preview passed in {time.perf_counter() - started:.1f}s")
    except Exception as exc:
        print(f"[FAIL] Live preview failed after {time.perf_counter() - started:.1f}s: {exc}")

if RUN_LIVE_AUDIO_SMOKE_TEST:
    started = time.perf_counter()
    try:
        audio_path = tell_story(sample_request, mode="excerpt", max_minutes=1)
        assert isinstance(audio_path, str) and Path(audio_path).is_file()
        display(Audio(filename=audio_path))
        print(f"[PASS] Live audio smoke test passed in {time.perf_counter() - started:.1f}s")
    except Exception as exc:
        print(f"[FAIL] Live audio smoke test failed after {time.perf_counter() - started:.1f}s: {exc}")

if not RUN_LIVE_PREVIEW_TEST and not RUN_LIVE_AUDIO_SMOKE_TEST:
    print("Live API smoke tests are disabled. Set a flag to True to run one deliberately.")

## 13. Direct function examples

Previewing avoids text-to-speech usage:

```python
script = preview_story(
    "Tell the complete story of The Happy Prince",
    mode="retelling",
    max_minutes=5,
)
```

Generate a faithful multi-voice passage:

```python
audio_path = tell_story(
    "From The Happy Prince, read the scene where the Swallow first speaks with the Prince",
    mode="excerpt",
    max_minutes=3,
)
print(audio_path)
```

A complete exact reading is intentionally rejected; use a focused excerpt or a condensed retelling.

## 14. Storyteller checklist

- [x] Standalone Colab notebook; the RAG/exam notebook is not modified.
- [x] PDF validation, extractable text, pages, paragraphs, and dialogue punctuation are preserved.
- [x] Story boundaries and character voice roles are extracted from the PDF.
- [x] Story/character catalogs and embeddings use fingerprinted caches.
- [x] Semantic passage selection never crosses story boundaries.
- [x] Faithful excerpts and clearly labelled grounded retellings are supported.
- [x] Character-specific OpenAI voice override and built-in role fallback are implemented.
- [x] OpenAI 	ts-1 generates every speech segment using built-in voices.
- [x] Segment progress, pauses, normalization, concatenation, inline playback, and MP3 saving are implemented.
- [x] uild_story_catalog, preview_story, and 	ell_story public functions exist.
- [x] Function and widget interfaces are included.
- [x] Local validation and opt-in live smoke tests are included.
- [x] Only OPENAI_API_KEY is required; custom voice overrides are not displayed.